# Herramientas · Criptografía

Firma digital y cifrado asimétrico con la librería `cryptography` (ejercicios de [El Libro De Python](https://ellibrodepython.com/ejemplos-ejercicios-python)).

## 1. Firma digital con otra clave pública

Firmar un mensaje con una clave privada de curva elíptica y verificarlo con su clave pública. Después, intentar verificar la firma con una clave pública distinta y explicar qué sucede.

La verificación falla (`InvalidSignature`): la firma solo la puede validar la clave pública emparejada con la privada que firmó, así que otra clave no puede dar por buena una firma que no hizo su pareja.

In [1]:
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec

privada = ec.generate_private_key(ec.SECP256R1())
publica = privada.public_key()
otra_publica = ec.generate_private_key(ec.SECP256R1()).public_key()
mensaje = b"Envia 1 Euro al IBAN 123"
firma = privada.sign(mensaje, ec.ECDSA(hashes.SHA256()))

def verifica(clave, firma, mensaje):
    try:
        clave.verify(firma, mensaje, ec.ECDSA(hashes.SHA256()))
        return "Verificación OK"
    except InvalidSignature:
        return "Verificación NOK"

print(verifica(publica, firma, mensaje))
print(verifica(publica, firma, b"Envia 100 Euros al IBAN 456"))
print(verifica(otra_publica, firma, mensaje))

Verificación OK
Verificación NOK
Verificación NOK


## 2. Descifrar con otra clave privada

Cifrar un mensaje con la clave pública RSA del destinatario y descifrarlo con su clave privada. Después, intentar descifrarlo con otra clave privada y explicar qué sucede.

El descifrado falla (`ValueError`): solo la clave privada emparejada con la pública usada para cifrar puede recuperar el mensaje. El relleno OAEP detecta que el resultado no es válido en lugar de devolver datos basura.

In [2]:
from cryptography.hazmat.primitives.asymmetric import padding, rsa

privada = rsa.generate_private_key(public_exponent=65537, key_size=2048)
publica = privada.public_key()
otra_privada = rsa.generate_private_key(public_exponent=65537, key_size=2048)
relleno = padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)

cifrado = publica.encrypt(b"Mensaje secreto de El Libro de Python", relleno)
print(privada.decrypt(cifrado, relleno))
try:
    otra_privada.decrypt(cifrado, relleno)
except ValueError as e:
    print("ValueError:", e)

b'Mensaje secreto de El Libro de Python'
ValueError: Decryption failed
